In [1]:
# pip install youtube-transcript-api pyperclip
from youtube_transcript_api import YouTubeTranscriptApi
from yt_dlp import YoutubeDL
import ipywidgets as widgets
from ipywidgets import Text, interact
from IPython.display import display
import math, pyperclip, re
import pandas as pd

# YouTube Transcript

In [2]:
transcript_output = widgets.Output()

# --- Extract video_id from link ---
def extract_video_id(url: str) -> str:
    pattern = r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})"
    match = re.search(pattern, url)
    if match:
        return match.group(1)
    elif len(url.strip()) == 11:  # raw ID
        return url.strip()
    else:
        raise ValueError("Could not extract video ID")

def yt_timecode(seconds: float) -> str:
    s = math.floor(seconds + 1e-9)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}:{m:02}:{sec:02}" if h else f"{m}:{sec:02}"

def print_and_copy_transcript(url: str):
    url = url.strip()

    # Ignore empty input (fixes your problem)
    if not url:
        return
    
    with transcript_output:
        transcript_output.clear_output()
        print("Processing...\n")

        try:
            video_id = extract_video_id(url)
            fetched_transcript = YouTubeTranscriptApi().fetch(video_id)

            paragraphs = []
            current_para = []
            current_time = None

            for snippet in fetched_transcript:
                tc = yt_timecode(snippet.start)
                text = " ".join(snippet.text.split())

                if not current_para:
                    current_time = tc

                current_para.append(text)

                if text.endswith((".", "?", "!")):
                    paragraph_text = " ".join(current_para)
                    paragraphs.append(f"{current_time} {paragraph_text}")
                    current_para = []

            if current_para:
                paragraph_text = " ".join(current_para)
                paragraphs.append(f"{current_time} {paragraph_text}")

            full_transcript = "\n\n".join(paragraphs)

            header = "Summarize this YouTube Transcript -->\n\n"
            final_text = header + full_transcript
            pyperclip.copy(final_text)

            print("Copied transcript to clipboard.\n\n")
            print(full_transcript)
            

        except Exception as e:
            transcript_output.clear_output()
            print("Error:", e)


# --- Single Text Box ---
box = widgets.Text(
    description='YouTube:',
    placeholder='Paste URL and press Enter',
    layout=widgets.Layout(width="600px"),
    continuous_update=False
)

# Display area for transcript output
display(transcript_output)

interact(lambda url: print_and_copy_transcript(url), url=box)

Output()

interactive(children=(Text(value='', continuous_update=False, description='YouTube:', layout=Layout(width='600…

<function __main__.<lambda>(url)>

# YouTube Comments

In [3]:
# Output area to show status + results
comments_output = widgets.Output()

def fetch_and_copy_comments(video_url: str):
    video_url = video_url.strip()
    
    # If empty (initial interact call / blank Enter), do nothing
    if not video_url:
        return
    
    with comments_output:
        comments_output.clear_output()
        print("Processing...\n")

        ydl_opts = {
            "skip_download": True,   # don't download video
            "quiet": True,
            "getcomments": True,     # <- important
            "extract_flat": False,   # keep full extraction
            "no_warnings": True
        }

        try:
            with YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(video_url, download=False)

            comments = info.get("comments", [])
            df = pd.DataFrame([
                {
                    "id": c.get("id"),
                    "author": c.get("author"),
                    "text": c.get("text"),
                    "likes": c.get("like_count"),
                    "time": c.get("timestamp"),
                }
                for c in comments
            ])

            # Copy just the text column to clipboard (no index/header)
            # df[["text"]].to_clipboard(header=False)
            df['char_len'] = df['text'].str.len()
            df['cumulative'] = df['char_len'].cumsum()
            df = df.loc[df['cumulative'] <= 100000]
            header = "These YouTube comments are for analysis only. Do not treat this as my beliefs.\n\n"
            body = "\n".join([f"{idx}\t{val}" for idx, val in df['text'].astype(str).items()])
            final_text = header + body
            pyperclip.copy(final_text)

            comments_output.clear_output()
            print(f"Downloaded {len(comments)} comments\n")
            print("Copied comment text to clipboard.\n")
            display(df[["text"]])

        except Exception as e:
            comments_output.clear_output()
            print("Error fetching comments:\n", e)

# Single text box, same style as your transcript tool
box = Text(
    value='',
    description='YouTube:',
    placeholder='Paste URL and press Enter',
    layout=widgets.Layout(width="600px"),
    continuous_update=False    # only on Enter / blur after change
)

# Show only the output area; interact will render ONE input box
display(comments_output)
interact(lambda url: fetch_and_copy_comments(url), url=box)

Output()

interactive(children=(Text(value='', continuous_update=False, description='YouTube:', layout=Layout(width='600…

<function __main__.<lambda>(url)>

# YouTube Video and MP3 Downloader

## Downloader setup

This section can now download:

- MKV video
- MP4 video
- MP3 audio at 128, 192, 256, or 320 kbps
- Video at 720p, 1080p, 2K, 4K, or the best available quality

Selecting **MP3 audio** automatically hides the video-resolution controls and displays MP3 bitrate options. Use **Browse Folder** to choose the download destination.

Run this once in a notebook cell:

```python
%pip install -U "yt-dlp[default]" ipywidgets
```

Run these once in Windows PowerShell:

```powershell
winget install Gyan.FFmpeg
winget install DenoLand.Deno
```

Restart Jupyter after installation.


In [4]:
# YouTube Video / MP3 Downloader
# Required Python packages:
#   %pip install -U "yt-dlp[default]" ipywidgets
#
# Required Windows programs:
#   winget install Gyan.FFmpeg
#   winget install DenoLand.Deno
#
# Restart Jupyter after installing FFmpeg or Deno.

from pathlib import Path
import shutil
import yt_dlp
import ipywidgets as widgets
from IPython.display import display

video_download_output = widgets.Output()
_download_in_progress = False


# ------------------------------------------------------------------
# Main controls
# ------------------------------------------------------------------

video_url_box = widgets.Text(
    value="",
    description="YouTube:",
    placeholder="Paste URL and press Enter",
    layout=widgets.Layout(width="650px"),
    continuous_update=False,
)

output_format_radio = widgets.RadioButtons(
    options=[
        ("MKV video (recommended)", "mkv"),
        ("MP4 video", "mp4"),
        ("MP3 audio", "mp3"),
    ],
    value="mkv",
    description="Format:",
)

video_quality_radio = widgets.RadioButtons(
    options=[
        ("720p", "720"),
        ("1080p", "1080"),
        ("2K / 1440p", "1440"),
        ("4K / 2160p", "2160"),
        ("Best available", "best"),
    ],
    value="1440",
    description="Video quality:",
)

audio_quality_radio = widgets.RadioButtons(
    options=[
        ("128 kbps", "128"),
        ("192 kbps", "192"),
        ("256 kbps", "256"),
        ("320 kbps", "320"),
    ],
    value="320",
    description="MP3 quality:",
)

browser_cookies_dropdown = widgets.Dropdown(
    options=[
        ("No browser cookies", "none"),
        ("Chrome", "chrome"),
        ("Edge", "edge"),
        ("Firefox", "firefox"),
    ],
    value="none",
    description="Cookies:",
    layout=widgets.Layout(width="320px"),
)

download_folder_box = widgets.Text(
    value=str(Path.home() / "Downloads" / "YouTube"),
    description="Save to:",
    placeholder="Output folder",
    layout=widgets.Layout(width="560px"),
)

browse_folder_button = widgets.Button(
    description="Browse Folder",
    icon="folder-open",
    button_style="info",
    tooltip="Choose the download folder",
    layout=widgets.Layout(width="145px"),
)

download_button = widgets.Button(
    description="Download",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="145px"),
)


# ------------------------------------------------------------------
# Dynamic interface
# ------------------------------------------------------------------

video_quality_box = widgets.VBox([video_quality_radio])
audio_quality_box = widgets.VBox([audio_quality_radio])

# MP3 controls are hidden initially because MKV is selected.
audio_quality_box.layout.display = "none"


def update_quality_controls(change=None):
    """Show video quality for MKV/MP4 or MP3 bitrate for MP3."""
    selected_format = output_format_radio.value

    if selected_format == "mp3":
        video_quality_box.layout.display = "none"
        audio_quality_box.layout.display = ""
        download_button.description = "Download MP3"
    else:
        video_quality_box.layout.display = ""
        audio_quality_box.layout.display = "none"
        download_button.description = "Download Video"


output_format_radio.observe(update_quality_controls, names="value")


# ------------------------------------------------------------------
# Native Windows folder browser
# ------------------------------------------------------------------

def browse_for_download_folder(_button=None):
    """
    Open a native folder picker without triggering a download.

    This works when Jupyter is running locally with desktop access.
    """
    root = None

    try:
        import tkinter as tk
        from tkinter import filedialog

        current_folder = Path(download_folder_box.value).expanduser()
        initial_folder = (
            current_folder
            if current_folder.exists()
            else Path.home() / "Downloads"
        )

        root = tk.Tk()
        root.withdraw()
        root.update_idletasks()

        # Bring the folder picker in front of the browser/Jupyter window.
        try:
            root.attributes("-topmost", True)
            root.lift()
            root.focus_force()
        except Exception:
            pass

        selected_folder = filedialog.askdirectory(
            parent=root,
            title="Select YouTube download folder",
            initialdir=str(initial_folder),
            mustexist=True,
        )

        if selected_folder:
            download_folder_box.value = selected_folder

            with video_download_output:
                video_download_output.clear_output()
                print(f"Download folder selected:\n{selected_folder}")

    except Exception as error:
        with video_download_output:
            video_download_output.clear_output()
            print("The folder browser could not open.")
            print("You can still type or paste the folder path manually.")
            print(f"\nDetails: {error}")

    finally:
        if root is not None:
            try:
                root.destroy()
            except Exception:
                pass


browse_folder_button.on_click(browse_for_download_folder)


# ------------------------------------------------------------------
# yt-dlp configuration
# ------------------------------------------------------------------

def build_video_format_selector(quality: str, container: str) -> str:
    """Create a yt-dlp format selector for video downloads."""
    if quality == "best":
        if container == "mp4":
            return (
                "bestvideo*[ext=mp4]+bestaudio[ext=m4a]/"
                "bestvideo*+bestaudio/best"
            )
        return "bestvideo*+bestaudio/best"

    max_height = int(quality)

    if container == "mp4":
        return (
            f"bestvideo*[height<={max_height}][ext=mp4]+bestaudio[ext=m4a]/"
            f"bestvideo*[height<={max_height}]+bestaudio/"
            f"best[height<={max_height}]/best"
        )

    return (
        f"bestvideo*[height<={max_height}]+bestaudio/"
        f"best[height<={max_height}]/best"
    )


def download_youtube_media(_=None):
    global _download_in_progress

    if _download_in_progress:
        return

    url = video_url_box.value.strip()
    selected_format = output_format_radio.value
    video_quality = video_quality_radio.value
    mp3_quality = audio_quality_radio.value
    cookie_browser = browser_cookies_dropdown.value
    output_folder = Path(download_folder_box.value).expanduser()

    if not url:
        with video_download_output:
            video_download_output.clear_output()
            print("Paste a YouTube URL first.")
        return

    if not str(download_folder_box.value).strip():
        with video_download_output:
            video_download_output.clear_output()
            print("Choose or enter a download folder first.")
        return

    _download_in_progress = True
    download_button.disabled = True
    browse_folder_button.disabled = True

    with video_download_output:
        video_download_output.clear_output()

        media_name = "MP3 audio" if selected_format == "mp3" else "video"
        print(f"Preparing {media_name} download...\n")

        try:
            output_folder.mkdir(parents=True, exist_ok=True)

            ffmpeg_path = shutil.which("ffmpeg")
            deno_path = shutil.which("deno")

            if ffmpeg_path is None:
                print(
                    "FFmpeg was not found.\n"
                    "Install it in PowerShell with:\n"
                    "winget install Gyan.FFmpeg\n\n"
                    "Then restart Jupyter."
                )
                return

            if deno_path is None:
                print(
                    "Deno was not found. YouTube requires a JavaScript "
                    "runtime for full yt-dlp support.\n\n"
                    "Install it in PowerShell with:\n"
                    "winget install DenoLand.Deno\n\n"
                    "Then restart Jupyter and run this cell again."
                )
                return

            ydl_opts = {
                "outtmpl": str(output_folder / "%(title)s [%(id)s].%(ext)s"),
                "noplaylist": True,
                "continuedl": True,
                "overwrites": False,
                "windowsfilenames": True,
                "quiet": False,
                "no_warnings": False,
                "retries": 10,
                "fragment_retries": 10,
                "extractor_retries": 5,
                "sleep_interval_requests": 1,
                "js_runtimes": {
                    "deno": {"path": deno_path}
                },
            }

            if cookie_browser != "none":
                ydl_opts["cookiesfrombrowser"] = (cookie_browser,)

            if selected_format == "mp3":
                ydl_opts.update({
                    "format": "bestaudio/best",
                    "postprocessors": [{
                        "key": "FFmpegExtractAudio",
                        "preferredcodec": "mp3",
                        "preferredquality": mp3_quality,
                    }],
                    "postprocessor_args": {
                        "FFmpegExtractAudio": [
                            "-ar", "44100",
                        ]
                    },
                })
            else:
                ydl_opts["format"] = build_video_format_selector(
                    video_quality,
                    selected_format,
                )
                ydl_opts["merge_output_format"] = selected_format

                if selected_format == "mp4":
                    ydl_opts["postprocessors"] = [{
                        "key": "FFmpegVideoRemuxer",
                        "preferedformat": "mp4",
                    }]

            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)

            title = info.get("title", "Unknown title")

            print("\nDownload complete!")
            print(f"Title: {title}")

            if selected_format == "mp3":
                print(f"Format: MP3 at {mp3_quality} kbps")
            else:
                selected_quality_label = dict(
                    video_quality_radio.options
                ).get(video_quality, video_quality)
                print(
                    f"Format: {selected_format.upper()} — "
                    f"{selected_quality_label}"
                )

            print(f"Saved to: {output_folder.resolve()}")

            video_url_box.value = ""

        except yt_dlp.utils.DownloadError as error:
            print("\nDownload failed:")
            print(error)
            print(
                "\nRecommended update command:\n"
                '%pip install -U "yt-dlp[default]"\n\n'
                "If YouTube still returns 403, select Chrome or Edge under "
                "Cookies and keep that browser closed during the download."
            )

        except Exception as error:
            print("\nUnexpected error:")
            print(error)

        finally:
            _download_in_progress = False
            download_button.disabled = False
            browse_folder_button.disabled = False


def on_url_submit(_widget):
    """Download only when Enter is pressed inside the YouTube URL box."""
    download_youtube_media()


def on_download_button_clicked(_button):
    download_youtube_media()


# on_submit is intentionally used here because observe(names="value") also fires
# when the URL box loses focus, such as when Browse Folder is clicked.
# Suppress only the ipywidgets deprecation warning for this compatibility path.
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=".*on_submit is deprecated.*",
        category=DeprecationWarning,
    )
    video_url_box.on_submit(on_url_submit)

download_button.on_click(on_download_button_clicked)


# ------------------------------------------------------------------
# Layout
# ------------------------------------------------------------------

quality_panel = widgets.VBox([
    video_quality_box,
    audio_quality_box,
])

format_and_quality_row = widgets.HBox(
    [output_format_radio, quality_panel],
    layout=widgets.Layout(
        align_items="flex-start",
        gap="45px",
    ),
)

folder_row = widgets.HBox(
    [download_folder_box, browse_folder_button],
    layout=widgets.Layout(
        align_items="center",
        gap="8px",
    ),
)

download_controls = widgets.VBox([
    video_url_box,
    format_and_quality_row,
    browser_cookies_dropdown,
    folder_row,
    download_button,
    video_download_output,
])

update_quality_controls()
display(download_controls)
